# Using `SyntheticsDownloader` to download Instaseis synthetics

This example demonstrates how to download Instaseis synthetics using the class contained in the DAE directory.

A class to handle downloading synthetics is available as `DAE.SyntheticsDownloader`, which uses Instaseis to access precomputed Green's function databases in order to generate synthetic seismograms for 1D Earth models. Source-receiver geometries and moment tensors are generated randomly based on inputs. Documentation for this class be found in `DAE/synthetics.py`, but some key considerations and use cases are covered here.

Further information about Instaseis is available here: https://instaseis.net/.

In [ ]:
import sys
import glob
import numpy as np
import matplotlib.pyplot as plt

# The DAE directory can be imported as a package.
import DAE

Choices about the training synthetics are made when initialising `SyntheticsDownloader`.

One of the most important decisions to make is which Instaseis database to use. We can either use a locally stored Green's function database (calculated using AxiSEM or AxiSEM3D), or one of the precomputed databases hosted on Syngine (https://ds.iris.edu/ds/products/syngine/). For simplicity we will use a Syngine database here.

In [ ]:
database = 'prem_a_2s'

We can also decide what phases and components we wish to be in our synthetics. The code randomly selects a phase which will be in the centre of each trace. Phases follow the naming conventions used by ObsPy TauP (https://docs.obspy.org/packages/obspy.taup.html) and components are a single string separated by commas. If we want to specify a certain distance range or source depth range we can do this.

In [ ]:
phase_list = ['Sdiff']
components = 'T'
minimum_distance = 105.
maximum_distance = 125.
minimum_source_depth = 300.
maximum_source_depth = 500.

If we want our synthetics to be filtered to a specific period range then we can specifiy this. If these periods are not passed to `SyntheticsDownloader` then the synthetics will be filtered only to the periods required to make the Instaseis sythetics valid.

In [ ]:
minimum_period = 10.
maximum_period = 20.

Finally we can make some choices about the files that we want the synthetics to be stored in - the DAE takes traces stored in 2D NumPy arrays in .npz files. The size of the files and parameters for the time axis of the traces should be consistent with any accompaying noise datasets downloaded with `NoiseDownloader`.

In [ ]:
output_directory = './Data/Example_Synthetics/'
traces_per_file = 6
length_of_trace_in_samples = 6001
sampling_rate_in_Hz = 20

All of these options are passed to the downloader during initialisation.

In [ ]:
sd = DAE.SyntheticsDownloader(traces_per_file=traces_per_file,
                              length_of_trace_in_samples=length_of_trace_in_samples,
                              sampling_rate_in_Hz=sampling_rate_in_Hz,
                              minimum_distance=minimum_distance, 
                              maximum_distance=maximum_distance,
                              minimum_source_depth=minimum_source_depth, 
                              maximum_source_depth=maximum_source_depth,
                              minimum_period=minimum_period,
                              maximum_period=maximum_period,
                              phase_list=phase_list,
                              components=components, 
                              database=database,
                              output_directory=output_directory)

The `build_dataset` method runs the download and a progressbar keeps track. It is at this point that we say how many traces we want to download.

In [ ]:
number_of_traces_to_download = 18
sd.build_dataset(number_of_traces=number_of_traces_to_download)

Files in .npz format containing the downloaded traces should now be in the `output_directory`.

In [ ]:
file_list = glob.glob(f'{output_directory}/*.npz')
print(file_list)

We can easily plot the contents of the files and see these traces.

In [ ]:
def plot_figure(file_list):
    # Set up the figure. Lets put each file into a different subplot
    fig, axes = plt.subplots(nrows=1, ncols=len(file_list))
    for ax, fname in zip(axes, file_list):

        # Read in the .npz file
        data = np.load(fname)['data']

        # Plot each trace
        for i, trace in enumerate(data):
            ax.plot(i + 0.5 * trace / np.max(np.abs(trace)), 'k', linewidth=0.75)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xlim([0, data.shape[1]])
        ax.set_title(fname.split('/')[-1])
    plt.show()

# Call the function
plot_figure(file_list)

If we want do not want our file numbering to start from zero then we can pass `startcount` to `build_dataset`, which is the number of the file that we want to start from. This allows parallel downloads to be run without overwriting, or for existing datasets to be expanded.

In this example we will download one more file.

In [ ]:
number_of_downloaded_files = len(file_list)
sd.build_dataset(number_of_traces=6, startcount=number_of_downloaded_files)

In [ ]:
new_file_list = glob.glob(f'{output_directory}/*.npz')
plot_figure(new_file_list)